#### Get Marker Genes for All Datasets

In [ ]:
import anndata as ad
# adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/interim/single-cell/sc_integrated_withoutnans.h5ad')
adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/breast_xenium/interim/single-cell/breast_cancer_chromium_genesubset.h5ad')
# The integrated object has harmonized labels; the count check below requires raw counts in X.
adata.obs.cell_type_harmonized.value_counts()

cell_type_harmonized
T CD4+         9374
Macrophage     5714
B cells        5218
Malignant      3694
T CD8+         3383
DC             2942
Neutrophil     2728
Plasmablast    2197
NK             1116
Monocyte       1027
Mast            732
Epithelial      587
Fibroblast      585
Endothelial     535
Treg             19
Name: count, dtype: int64

In [20]:
cell_type_groups = {
    "Lymphoid": [
        "B cells",
        "Plasmablast",
        "NK",
    ],
    "T-cells": [
        "T CD4+",
        "T CD8+",
        "Treg",
    ],
    "Myeloid": [
        "DC",
        "Macrophage",
        "Monocyte",
        "Neutrophil",
    ],

    "Stromal": [
        "Endothelial",
        "Fibroblast",
    ],

    "Epithelial": [
        "Epithelial",
        "Malignant",
    ],

    "Mast": [
        "Mast",
    ],
}

## Patient-level pseudobulk marker genes with PyDESeq2

Set `ANALYSIS_MODE` to `flat` for cell-type vs all-other markers, or `hierarchical` for broad-group markers followed by subtype-vs-sibling markers within each group from `cell_type_groups`. The paired design (`~ patient + group`) uses the configured sample identifier—not cells—as replicates. Prefer `Patient` when libraries from the same patient are not independent. Positive log2 fold change means enrichment in the target. These are markers within the measured CosMx gene panel, not whole-transcriptome markers.

Requires `pydeseq2` in the notebook kernel (`pip install pydeseq2`).

In [21]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PATIENT_COL = 'Patient'  # Existing choice; use 'Patient' for biological independence.
CELL_TYPE_COL = 'cell_type_harmonized'
ANALYSIS_MODE = 'hierarchical'  # 'flat' or 'hierarchical'
COUNT_LAYER = None  # None means adata.X; use a raw-count layer if needed.
MIN_CELLS_PER_GROUP = 20
MIN_PATIENTS = 3
MIN_TOTAL_GENE_COUNT = 10
PADJ_CUTOFF = 0.01
MIN_LOG2_FC = 1.5
N_CPUS = 4
BASE_OUTPUT_DIR = Path('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/results/markers/pydeseq2')
if ANALYSIS_MODE not in {'flat', 'hierarchical'}:
    raise ValueError('ANALYSIS_MODE must be flat or hierarchical')
OUTPUT_DIR = BASE_OUTPUT_DIR if ANALYSIS_MODE == 'flat' else BASE_OUTPUT_DIR / 'hierarchical'

obs = adata.obs[[PATIENT_COL, CELL_TYPE_COL]].copy()
obs = obs.rename(columns={CELL_TYPE_COL:'cell_type',PATIENT_COL:'patient'})
obs.columns = ['patient', 'cell_type']
obs = obs.astype('string')
valid = obs.notna().all(axis=1) & obs.ne('').all(axis=1)
obs = obs.loc[valid].astype(str)
X = adata.layers[COUNT_LAYER] if COUNT_LAYER else adata.X
X = X[valid.to_numpy()]
if not adata.var_names.is_unique:
    raise ValueError('Gene names must be unique before DESeq2.')
values = X.data if sparse.issparse(X) else np.asarray(X)
if not np.isfinite(values).all() or (values < 0).any() or not np.allclose(values, np.rint(values)):
    raise ValueError('PyDESeq2 requires nonnegative raw integer counts; select a raw-count layer.')

# One row per patient/cell type, summing counts across its cells.
group_ids, groups = pd.factorize(pd.MultiIndex.from_frame(obs), sort=True)
membership = sparse.csr_matrix(
    (np.ones(len(group_ids), dtype=np.int64), (group_ids, np.arange(len(group_ids)))),
    shape=(len(groups), len(group_ids)),
)
pseudobulk = membership @ X
pseudobulk = pseudobulk.toarray() if sparse.issparse(pseudobulk) else np.asarray(pseudobulk)
pseudobulk = np.rint(pseudobulk).astype(np.int64)
group_metadata = groups.to_frame(index=False).reset_index(drop=True)
group_metadata['n_cells'] = np.bincount(group_ids, minlength=len(groups))
group_metadata = group_metadata.rename(columns={0:'patient',1:'cell_type'})
genes = adata.var_names.astype(str).to_numpy()
print(f'{len(obs):,} cells -> {len(groups):,} patient/cell-type pseudobulks; {len(genes):,} genes')
display(group_metadata.pivot(index='patient', columns='cell_type', values='n_cells').fillna(0).astype(int))

39,851 cells -> 104 patient/cell-type pseudobulks; 956 genes


cell_type,B cells,DC,Endothelial,Epithelial,Fibroblast,Macrophage,Malignant,Mast,Monocyte,NK,Neutrophil,Plasmablast,T CD4+,T CD8+,Treg
patient,,,,,,,,,,,,,,,
p1,248,746,58,122,225,1616,1326,165,397,286,690,286,1331,749,5
p2,26,22,4,2,12,37,116,7,16,6,15,112,83,20,1
p3,3268,677,233,286,189,479,539,218,184,362,696,410,2220,333,4
p4,155,328,12,7,2,398,181,46,11,181,29,3,2465,1086,5
p5,716,298,92,30,85,675,776,30,22,83,24,289,958,365,2
p6,698,381,72,51,49,575,230,62,36,97,52,818,1836,696,0
p7,107,490,64,89,23,1934,526,204,361,101,1222,279,481,134,2


In [22]:
cell_types = sorted(group_metadata['cell_type'].unique())
# ANALYSIS_MODE ='flat'
if ANALYSIS_MODE == 'hierarchical':
    flattened = [ct for members in cell_type_groups.values() for ct in members]
    if len(flattened) != len(set(flattened)) or set(flattened) != set(cell_types):
        raise ValueError(f'cell_type_groups must cover each observed type exactly once; missing={set(cell_types)-set(flattened)}, extra={set(flattened)-set(cell_types)}')
    comparisons = [
        {'stage': 'broad', 'parent_group': '__all__', 'cell_type': group,
         'target_types': members, 'pool_types': cell_types}
        for group, members in cell_type_groups.items()
    ]
    comparisons += [
        {'stage': 'subtype', 'parent_group': group, 'cell_type': ct,
         'target_types': [ct], 'pool_types': members}
        for group, members in cell_type_groups.items() if len(members) > 1 for ct in members
    ]
else:
    comparisons = [
        {'stage': 'flat', 'parent_group': '__all__', 'cell_type': ct,
         'target_types': [ct], 'pool_types': cell_types}
        for ct in cell_types
    ]

results, skipped = [], []
patients = group_metadata['patient'].unique()
for comparison in comparisons:
    target = comparison['cell_type']
    sample_counts, sample_rows = [], []
    for patient in patients:
        patient_mask = group_metadata['patient'].eq(patient).to_numpy()
        pool_mask = patient_mask & group_metadata['cell_type'].isin(comparison['pool_types']).to_numpy()
        target_mask = pool_mask & group_metadata['cell_type'].isin(comparison['target_types']).to_numpy()
        other_mask = pool_mask & ~target_mask
        n_target = int(group_metadata.loc[target_mask, 'n_cells'].sum())
        n_other = int(group_metadata.loc[other_mask, 'n_cells'].sum())
        if min(n_target, n_other) < MIN_CELLS_PER_GROUP:
            continue
        sample_counts.extend([pseudobulk[target_mask].sum(axis=0),
                              pseudobulk[other_mask].sum(axis=0)])
        sample_rows.extend([{'patient': patient, 'group': 'target', 'n_cells': n_target},
                            {'patient': patient, 'group': 'other', 'n_cells': n_other}])
    n_patients = len(sample_rows) // 2
    if n_patients < MIN_PATIENTS:
        skipped.append({**{k: comparison[k] for k in ('stage', 'parent_group', 'cell_type')},
                        'n_patients': n_patients, 'reason': 'too few paired patients'})
        continue
    metadata = pd.DataFrame(sample_rows)
    metadata.index = [f'pb_{i}' for i in range(len(metadata))]
    counts = pd.DataFrame(sample_counts, index=metadata.index, columns=genes)
    counts = counts.loc[:, counts.sum(axis=0) >= MIN_TOTAL_GENE_COUNT]
    if counts.empty:
        skipped.append({**{k: comparison[k] for k in ('stage', 'parent_group', 'cell_type')},
                        'n_patients': n_patients, 'reason': 'no genes pass count filter'})
        continue
    print(f"{comparison['stage']} / {target}: {n_patients} paired patients, {counts.shape[1]} genes")
    dds = DeseqDataSet(counts=counts, metadata=metadata, design='~ patient + group',
                       refit_cooks=True, n_cpus=N_CPUS, quiet=True)
    dds.deseq2()
    stats = DeseqStats(dds, contrast=['group', 'target', 'other'], quiet=True)
    stats.summary()
    result = stats.results_df.rename_axis('gene').reset_index()
    for key in ('stage', 'parent_group', 'cell_type'):
        result[key] = comparison[key]
    result['n_patients'] = n_patients
    results.append(result)

if not results:
    raise RuntimeError(f'No comparison had at least {MIN_PATIENTS} eligible paired patients. Skipped: {skipped}')
all_results = pd.concat(results, ignore_index=True)
candidate_markers = all_results.loc[
    all_results['padj'].lt(PADJ_CUTOFF) & all_results['log2FoldChange'].ge(MIN_LOG2_FC)
].sort_values(['stage', 'parent_group', 'cell_type', 'padj'], ascending=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_results.to_csv(OUTPUT_DIR / 'all_comparisons_deseq2.csv', index=False)
candidate_markers.to_csv(OUTPUT_DIR / 'candidate_markers_deseq2.csv', index=False)
pd.DataFrame(skipped, columns=['stage', 'parent_group', 'cell_type', 'n_patients', 'reason']).to_csv(OUTPUT_DIR / 'skipped_comparisons.csv', index=False)
print(f'Saved {len(candidate_markers):,} candidates across {len(results)} comparisons to {OUTPUT_DIR}')
display(candidate_markers.groupby(['stage', 'cell_type']).head(5))

broad / Lymphoid: 7 paired patients, 949 genes
broad / T-cells: 7 paired patients, 949 genes


/home/jwinchel/miniconda3/envs/deseq2/lib/python3.10/site-packages/pydeseq2/dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


broad / Myeloid: 7 paired patients, 949 genes


/home/jwinchel/miniconda3/envs/deseq2/lib/python3.10/site-packages/pydeseq2/dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


broad / Stromal: 5 paired patients, 948 genes


/home/jwinchel/miniconda3/envs/deseq2/lib/python3.10/site-packages/pydeseq2/dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


... (22 outputs left out)

subtype / Malignant: 5 paired patients, 881 genes


/home/jwinchel/miniconda3/envs/deseq2/lib/python3.10/site-packages/pydeseq2/dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


Saved 1,359 candidates across 19 comparisons to /work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/results/markers/pydeseq2/hierarchical


,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,stage,parent_group,cell_type,n_patients
3978,ENSG00000039068,860.719630,6.475141,0.291447,22.217226,2.340893e-109,2.221508e-106,broad,__all__,Epithelial,7
4313,ENSG00000175793,534.779936,7.920208,0.358210,22.110508,2.504363e-108,1.188320e-105,broad,__all__,Epithelial,7
4707,ENSG00000119888,722.174055,8.112293,0.384567,21.094615,8.912899e-99,2.819447e-96,broad,__all__,Epithelial,7
3886,ENSG00000189143,798.881766,8.288385,0.426326,19.441444,3.443375e-84,8.169407e-82,broad,__all__,Epithelial,7
4205,ENSG00000111057,848.021294,6.823455,0.362498,18.823428,4.853970e-79,9.212836e-77,broad,__all__,Epithelial,7
...,...,...,...,...,...,...,...,...,...,...,...
8972,ENSG00000145649,160.126234,3.526916,0.179206,19.680768,3.151913e-86,2.332416e-83,subtype,T-cells,T CD8+,7
9361,ENSG00000105374,92.340566,4.375226,0.260691,16.783157,3.241347e-63,1.199298e-60,subtype,T-cells,T CD8+,7
9315,ENSG00000153563,249.007462,3.841858,0.231340,16.607011,6.200721e-62,1.529511e-59,subtype,T-cells,T CD8+,7
9407,ENSG00000077984,317.907657,2.519401,0.156268,16.122320,1.778077e-58,3.289442e-56,subtype,T-cells,T CD8+,7


## Remove nonspecific markers

Starting from positive DESeq2 hits, retain genes that are significant for **only one target at the same hierarchy level** and detected substantially more often in that target than in its comparison set. Broad-group markers are checked against other broad groups; subtype markers are checked against siblings within their parent group. Detection is the fraction of cells with at least one raw count, calculated from the original single-cell matrix; it is not used for the DESeq2 p-values. The full and candidate results remain available separately.

In [24]:
MIN_TARGET_DETECTION_FRAC = 0.10
MAX_OTHER_TO_TARGET_DETECTION_RATIO = 0.50

# Aggregate detection (X > 0) by cell type, without densifying the cell-by-gene matrix.
type_ids, type_names = pd.factorize(obs['cell_type'], sort=True)
type_membership = sparse.csr_matrix(
    (np.ones(len(type_ids), dtype=np.int64), (type_ids, np.arange(len(type_ids)))),
    shape=(len(type_names), len(type_ids)),
)
detections = type_membership @ (X > 0)
detections = detections.toarray() if sparse.issparse(detections) else np.asarray(detections)
type_sizes = np.bincount(type_ids, minlength=len(type_names))
detection_fraction = pd.DataFrame(
    detections / type_sizes[:, None], index=type_names, columns=genes,
)

if ANALYSIS_MODE == 'hierarchical':
    broad_detection = pd.DataFrame({
        group: detections[np.isin(type_names, members)].sum(axis=0)
               / type_sizes[np.isin(type_names, members)].sum()
        for group, members in cell_type_groups.items()
    }, index=genes).T
else:
    broad_detection = None

def detection_for_comparison(stage, parent, target):
    if stage == 'broad':
        table = broad_detection
    elif stage == 'subtype':
        table = detection_fraction.loc[cell_type_groups[parent]]
    else:
        table = detection_fraction
    return table.loc[target], table.drop(index=target).max(axis=0)

marker_genes = candidate_markers.copy()
marker_genes['n_enriched_targets'] = marker_genes.groupby(
    ['stage', 'parent_group', 'gene']
)['cell_type'].transform('nunique')
detection_by_comparison = {
    key: detection_for_comparison(*key)
    for key in marker_genes[['stage', 'parent_group', 'cell_type']].itertuples(index=False, name=None)
}
marker_genes['target_detection_fraction'] = [
    detection_by_comparison[(stage, parent, target)][0].loc[gene]
    for stage, parent, target, gene in marker_genes[
        ['stage', 'parent_group', 'cell_type', 'gene']
    ].itertuples(index=False, name=None)
]
marker_genes['max_other_detection_fraction'] = [
    detection_by_comparison[(stage, parent, target)][1].loc[gene]
    for stage, parent, target, gene in marker_genes[
        ['stage', 'parent_group', 'cell_type', 'gene']
    ].itertuples(index=False, name=None)
]
marker_genes['other_to_target_detection_ratio'] = (
    marker_genes['max_other_detection_fraction']
    / marker_genes['target_detection_fraction'].replace(0, np.nan)
)
marker_genes = marker_genes.loc[
    marker_genes['n_enriched_targets'].eq(1)
    & marker_genes['target_detection_fraction'].ge(MIN_TARGET_DETECTION_FRAC)
    & marker_genes['other_to_target_detection_ratio'].le(MAX_OTHER_TO_TARGET_DETECTION_RATIO)
].copy()
marker_genes.to_csv(OUTPUT_DIR / 'cell_type_marker_genes_deseq2.csv', index=False)
print(f'Retained {len(marker_genes):,} specific markers of {len(candidate_markers):,} candidates')
display(marker_genes.groupby('cell_type').head(10))

Retained 541 specific markers of 1,359 candidates


,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,stage,parent_group,cell_type,n_patients,n_enriched_targets,target_detection_fraction,max_other_detection_fraction,other_to_target_detection_ratio
3978,ENSG00000039068,860.719630,6.475141,0.291447,22.217226,2.340893e-109,2.221508e-106,broad,__all__,Epithelial,7,1,0.465545,0.018049,0.038769
4313,ENSG00000175793,534.779936,7.920208,0.358210,22.110508,2.504363e-108,1.188320e-105,broad,__all__,Epithelial,7,1,0.348984,0.002981,0.008543
4707,ENSG00000119888,722.174055,8.112293,0.384567,21.094615,8.912899e-99,2.819447e-96,broad,__all__,Epithelial,7,1,0.434244,0.004098,0.009438
3886,ENSG00000189143,798.881766,8.288385,0.426326,19.441444,3.443375e-84,8.169407e-82,broad,__all__,Epithelial,7,1,0.415557,0.004351,0.010470
4205,ENSG00000111057,848.021294,6.823455,0.362498,18.823428,4.853970e-79,9.212836e-77,broad,__all__,Epithelial,7,1,0.529316,0.016964,0.032049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9657,ENSG00000172116,178.905199,4.032183,0.264594,15.239112,1.944952e-52,2.398774e-50,subtype,T-cells,T CD8+,7,1,0.339048,0.052632,0.155233
9655,ENSG00000100453,105.980942,3.357625,0.262043,12.813242,1.382359e-37,1.461351e-35,subtype,T-cells,T CD8+,7,1,0.160213,0.052632,0.328510
9159,ENSG00000180644,86.024401,2.639343,0.207592,12.714087,4.938462e-37,4.568077e-35,subtype,T-cells,T CD8+,7,1,0.199527,0.035524,0.178040
9573,ENSG00000100450,38.691220,3.856587,0.320360,12.038288,2.235476e-33,1.654252e-31,subtype,T-cells,T CD8+,7,1,0.102572,0.006401,0.062402


In [33]:
sorted(marker_genes['cell_type'].unique()),sorted(adata.obs['cell_type'].unique())
print(marker_genes.groupby(['cell_type']).size().reset_index(name='n_marker_genes').to_string())

      cell_type  n_marker_genes
0       B cells              19
1            DC               8
2   Endothelial              75
3    Epithelial              71
4    Fibroblast              71
5      Lymphoid              13
6    Macrophage              27
7     Malignant               1
8          Mast              16
9      Monocyte               5
10      Myeloid              52
11           NK              46
12   Neutrophil               8
13  Plasmablast              29
14      Stromal              68
15       T CD8+              15
16      T-cells              17


In [ ]:
marker_genes[marker_genes['cell_type']=='']

,gene,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,stage,parent_group,cell_type,n_patients,n_enriched_targets,target_detection_fraction,max_other_detection_fraction,other_to_target_detection_ratio
13164,ENSG00000163464,127.878544,7.458534,0.477811,15.609807,6.242511e-55,4.182482e-52,subtype,Myeloid,Neutrophil,6,1,0.190982,0.012658,0.066280
12890,ENSG00000180871,448.984215,5.240506,0.391768,13.376556,8.290000e-41,7.934714e-39,subtype,Myeloid,Neutrophil,6,1,0.338710,0.057403,0.169475
13389,ENSG00000182885,87.884663,6.110369,0.509283,11.997984,3.640569e-33,2.439181e-31,subtype,Myeloid,Neutrophil,6,1,0.203079,0.026290,0.129458
13165,ENSG00000115590,218.792112,4.691042,0.451546,10.388849,2.786945e-25,1.098384e-23,subtype,Myeloid,Neutrophil,6,1,0.293255,0.109056,0.371879
12836,ENSG00000163421,169.010221,6.513724,0.728732,8.938435,3.947214e-19,7.147657e-18,subtype,Myeloid,Neutrophil,6,1,0.166422,0.066212,0.397857
12780,ENSG00000163993,171.845311,5.815330,0.721279,8.062528,7.473234e-16,1.001413e-14,subtype,Myeloid,Neutrophil,6,1,0.211877,0.033106,0.156252
13404,ENSG00000255398,99.143939,3.424013,0.594158,5.762804,8.272808e-09,4.905116e-08,subtype,Myeloid,Neutrophil,6,1,0.190616,0.043402,0.227694
13291,ENSG00000182782,116.902695,2.237714,0.675899,3.310723,9.305543e-04,2.388779e-03,subtype,Myeloid,Neutrophil,6,1,0.204545,0.084179,0.411543


In [29]:
adata.obs.groupby(['cell_type_harmonized']).size()

cell_type_harmonized
B cells        5218
DC             2942
Endothelial     535
Epithelial      587
Fibroblast      585
Macrophage     5714
Malignant      3694
Mast            732
Monocyte       1027
NK             1116
Neutrophil     2728
Plasmablast    2197
T CD4+         9374
T CD8+         3383
Treg             19
dtype: int64